# LangChain-LangGraph-101

This hands-on notebook gives you a fast tour of core concepts using LangChain v1 and LangGraph v1:
- Prompt Templates
- Chat Models (OpenAI)
- Chaining / LCEL (basic)
- Structured Output
- Tool Calling
- Workflows with LangGraph
- LangChain Agents built on LangGraph


## 1) Setup



From the project directory, run `uv sync --locked`, copy `.env.example` to `.env`, and set `OPENAI_API_KEY`.
Launch with `uv run jupyter lab` and select the project's `.venv` Python environment.
Run the cells in order. Model calls use your OpenAI account and may incur API charges.

An existing shell environment variable takes precedence over `.env`.
Set `OPENAI_MODEL` in `.env` to override the example model (`gpt-4o-mini`).


![Setup: prepare the model connection](./notebook_images/01_setup.png)

*The setup check rejects missing keys and known placeholder values; it does not authenticate the key with OpenAI.*

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv(dotenv_path=".env")

api_key = os.getenv("OPENAI_API_KEY", "").strip()
if not api_key or api_key in {"sk-...", "your-openai-api-key", "your-api-key-here", "your_openai_api_key_here"}:
    raise ValueError("Set OPENAI_API_KEY in .env or your shell before running the notebook.")

MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-4o-mini").strip() or "gpt-4o-mini"


## 2) Prompt Templates

LangChain helps you compose prompts in a structured, reusable way.
We'll use `ChatPromptTemplate` to create a multi-turn style prompt.


![From a reusable prompt to a model response](./notebook_images/02_prompt_and_model.png)

In [ ]:
from datetime import date

from langchain_core.prompts import ChatPromptTemplate

from prompts import DATE_ASSISTANT_SYSTEM_PROMPT

prompt = ChatPromptTemplate.from_messages([
    ("system", DATE_ASSISTANT_SYSTEM_PROMPT),
    ("human", "{user_query}"),
])

example_user_query = "What is tomorrow's date?"
messages = prompt.format_messages(
    user_query=example_user_query,
    todays_date=date.today().isoformat(),
)

messages


## 3) Chat Models (OpenAI)

We'll call OpenAI via `langchain-openai`. `MODEL_NAME` comes from the setup cell.
If you change it, choose an OpenAI chat model that supports tool calling and JSON Schema structured output.
Some models also require different generation settings, such as omitting `temperature`.


In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=MODEL_NAME, temperature=0)
response = llm.invoke(messages)
print(response.text)


## 4) Chaining with LCEL (LangChain Expression Language)

LCEL lets you pipe components together with `|` for clean, testable flows.


![LCEL: connect the steps with a pipe](./notebook_images/03_lcel.png)

In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()

print(chain.invoke({
    "user_query": example_user_query,
    "todays_date": date.today().isoformat(),
}))


## 5) Structured Output

Use a Pydantic schema and `with_structured_output` to request a typed result.
Here, OpenAI's JSON Schema mode enforces the schema and Pydantic validates the returned fields.
Both dates are required strings in ISO format; schema validation alone does not guarantee that the inferred dates are correct.


![Structured output: request a DateRange](./notebook_images/04_structured_output.png)

In [ ]:
from pydantic import BaseModel, Field

from prompts import DATE_EXTRACTOR_PROMPT_TEMPLATE


class DateRange(BaseModel):
    start_date: str = Field(description="Start date in YYYY-MM-DD format")
    end_date: str = Field(description="End date in YYYY-MM-DD format")


structured_llm = llm.with_structured_output(
    DateRange, method="json_schema", strict=True
)
date_prompt = ChatPromptTemplate.from_messages([
    ("system", DATE_EXTRACTOR_PROMPT_TEMPLATE),
    ("human", "{user_query}"),
])
date_chain = date_prompt | structured_llm

user_query = "I need time off for the next 4 days"
todays_date = date.today().isoformat()
llm_response = date_chain.invoke({
    "todays_date": todays_date,
    "user_query": user_query,
})

llm_response


## 6) Tool Calling

Models can request calls to functions (tools) you expose.
We'll define a calculator tool, execute any requested calls, and send the tool results back to the model for a final answer.
This example handles one round of tool calls; the agent in section 8 manages repeated rounds automatically.


![Tool calling: the model asks, Python executes](./notebook_images/05_tool_calling.png)

In [ ]:
import math

from langchain.tools import tool
from langchain_core.messages import HumanMessage


@tool
def circle_area(radius: float) -> float:
    """Return the area of a circle for a given radius."""
    return math.pi * radius * radius


tools = [circle_area]
tools_by_name = {item.name: item for item in tools}
llm_with_tools = llm.bind_tools(tools)

msg = HumanMessage(content="I have a circular garden of radius 3. What's the area?")
ai_msg = llm_with_tools.invoke([msg])
print("Tool calls:", ai_msg.tool_calls)

conversation = [msg, ai_msg]
for tool_call in ai_msg.tool_calls:
    # Passing the whole ToolCall produces a ToolMessage with the matching call ID.
    tool_message = tools_by_name[tool_call["name"]].invoke(tool_call)
    conversation.append(tool_message)

# Use the unbound model to summarize the results without requesting more tools.
final = llm.invoke(conversation) if ai_msg.tool_calls else ai_msg
print(final.text)


## 7) LangGraph: Workflows

LangGraph builds **stateful** graphs for control flow (branching, loops, retries).
We'll classify the user's intent, route to a response node, and end the workflow.
`MessagesState` merges message updates, so each response node returns only its new message.
The intent is stored separately from the conversation.

![Nodes, state, and edges](./notebook_images/nodes_state_edges.png)


![Graph state: what each node reads and updates](./notebook_images/06_state_and_nodes.png)

In [ ]:
from typing import Literal, NotRequired

from langchain_core.messages import AIMessage
from langgraph.graph import END, START, MessagesState, StateGraph

from prompts import INTENT_DETECTION_TEMPLATE

IntentLabel = Literal["TIMEOFF", "GREETING", "OTHER"]


class GraphState(MessagesState):
    intent: NotRequired[IntentLabel]


class Intent(BaseModel):
    intent: IntentLabel = Field(description="The user's intent")


In [ ]:
# Define the classifier and graph nodes.
intent_prompt = ChatPromptTemplate.from_messages([
    ("system", INTENT_DETECTION_TEMPLATE),
    ("placeholder", "{messages}"),
])
intent_chain = intent_prompt | llm.with_structured_output(
    Intent, method="json_schema", strict=True
)


def intent_node(state: GraphState) -> dict:
    result = intent_chain.invoke({"messages": state["messages"]})
    return {"intent": result.intent}


def time_off_node(state: GraphState) -> dict:
    return {"messages": [AIMessage(content="Sure, I can help you with your time off request.")]}


def greeting_node(state: GraphState) -> dict:
    return {"messages": [AIMessage(content="Hello, how can I help you today?")]}


def other_node(state: GraphState) -> dict:
    return {"messages": [AIMessage(content="I'm sorry, I don't understand. Please try again.")]}


![Wire the workflow: classify, route, respond](./notebook_images/07_routing.png)

In [ ]:
graph = StateGraph(GraphState)
graph.add_node("intent_node", intent_node)
graph.add_node("time_off_node", time_off_node)
graph.add_node("greeting_node", greeting_node)
graph.add_node("other_node", other_node)


def router(state: GraphState) -> IntentLabel:
    return state["intent"]


graph.add_edge(START, "intent_node")
graph.add_conditional_edges("intent_node", router, {
    "TIMEOFF": "time_off_node",
    "GREETING": "greeting_node",
    "OTHER": "other_node",
})
for node_name in ("time_off_node", "greeting_node", "other_node"):
    graph.add_edge(node_name, END)

workflow = graph.compile()
workflow


![Run the workflow and read the last message](./notebook_images/08_run_workflow.png)

In [ ]:
user_query = "Hello"
# user_query = "I would like to apply for time off"
# user_query = "What is the weather like in SF?"

response = workflow.invoke({"messages": [HumanMessage(content=user_query)]})
response


In [ ]:
response["messages"][-1].text


## 8) LangChain: Agents Built on LangGraph

`langchain.agents.create_agent` builds a LangGraph-powered agent that calls tools until it can answer.
We'll give it two temperature conversion tools and stream the resulting messages.

Pass a checkpointer when **creating** the agent. Reusing a `thread_id` continues the same conversation;
a new ID starts a separate one. `InMemorySaver` retains state only while this Python process is running.

![Agent tool loop](./notebook_images/react.png)


![An agent loop with conversation memory](./notebook_images/09_agent_memory.png)

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver


@tool
def to_celsius(f: float) -> float:
    """Convert Fahrenheit to Celsius."""
    return (f - 32) * 5 / 9


@tool
def to_fahrenheit(c: float) -> float:
    """Convert Celsius to Fahrenheit."""
    return c * 9 / 5 + 32


memory = InMemorySaver()
agent = create_agent(
    model=llm,
    tools=[to_celsius, to_fahrenheit],
    system_prompt="Use the temperature conversion tools for conversions, then answer the user.",
    checkpointer=memory,
)
agent_config = {"configurable": {"thread_id": "temperature-demo"}}

events = agent.stream(
    {"messages": [HumanMessage(
        content="It's 77F outside. What's that in Celsius, rounded to 1 decimal?"
    )]},
    config=agent_config,
    stream_mode="values",
)
for event in events:
    event["messages"][-1].pretty_print()
